In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import AllChem

import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/Users/bikaschaudharytharu/python/ml-traning/data/preprocessed_bindingdb.csv")
df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,pKi
0,B(CC)(O)O,MGMRTVLTGLAGMLLGSMMPVQADMPRPTGLAADIRWTAYGVPHIR...,2.698970
1,B(CCCC)(O)O,MGMRTVLTGLAGMLLGSMMPVQADMPRPTGLAADIRWTAYGVPHIR...,3.259637
2,B(CCc1ccccc1)(O)O,RWTGRQKARGAATRARQKQRASLETMDKAVQRFRLQNPDLDSEALL...,5.886057
3,B([C@H](Cc1coc2c1cccc2)NC(=O)[C@@H]3C[C@H]4CC[...,MALASVLERPLPVNQRGFFGLGGRADLLDLGPGSLSDGLSLAAPGW...,5.617443
4,B([C@H](Cc1coc2c1cccc2)NC(=O)[C@@H]3C[C@H]4CC[...,MALLDVCGAPRGQRPESALPVAGSGRRSDPGHYSFSMRSPELALPR...,8.096910


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1925220 entries, 0 to 1925219
Data columns (total 3 columns):
 #   Column                             Dtype  
---  ------                             -----  
 0   Ligand SMILES                      str    
 1   BindingDB Target Chain Sequence 1  str    
 2   pKi                                float64
dtypes: float64(1), str(2)
memory usage: 44.1 MB


In [4]:
df.describe()

/Users/bikaschaudharytharu/python/Drug-target-interaction-using-machine-learning/venv/lib/python3.13/site-packages/pandas/core/nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,pKi
count,1.925220e+06
mean,inf
std,NaN
min,-4.000000e+00
25%,5.760200e+00
50%,6.853219e+00
75%,7.795880e+00
max,inf


In [5]:
print(df.isna().sum())

Ligand SMILES                        0
BindingDB Target Chain Sequence 1    0
pKi                                  0
dtype: int64


In [6]:
df.rename(columns={"BindingDB Target Chain Sequence 1": "Target Sequence"}, inplace=True)

In [7]:
df.dropna(subset=["Target Sequence", 'Ligand SMILES', 'pKi'], inplace=True)

In [8]:
def smiles_to_fingerprint(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
        return np.array(fp)
    else:
        return None

In [9]:
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_INDEX = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}

MAX_LEN = 1000

def encode_protine(seq):
    seq = seq[:MAX_LEN]
    encoded = [AA_TO_INDEX.get(a, 0) for a in seq]
    
    #padding
    if len(encoded) < MAX_LEN:
        encoded += [0] * (MAX_LEN - len(encoded))
    return encoded

In [10]:
ligand = []
proteins = []
labels = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    fp = smiles_to_fingerprint(row['Ligand SMILES'])

    if fp is None:
        continue

    prot = encode_protine(row['Target Sequence'])

    ligand.append(fp)
    proteins.append(prot)
    labels.append(row['pKi'])

  0%|          | 0/1925220 [00:00<?, ?it/s][13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DEPRECATION WARNING: please use MorganGenerator
[13:58:07] DE

KeyboardInterrupt: 

In [ ]:
X_ligand = np.array(ligand)
X_protein = np.array(proteins)
y = np.array(labels)

print("Dataset shapes:")
print("Ligand features:", X_ligand.shape)
print("Protein features:", X_protein.shape)
print("Labels:", y.shape)

